# Import libraries

In [1]:
import json
import math

In [2]:
import os

print(os.listdir())

['.anaconda', '.cache', '.conda', '.condarc', '.config', '.continuum', '.copilot', '.eclipse', '.hackerearth', '.idlerc', '.ipynb_checkpoints', '.ipython', '.jupyter', '.kaggle', '.keras', '.m2', '.matplotlib', '.ms-ad', '.p2', '.ssh', '.VirtualBox', '.vscode', '.vscode-shared', '1.bmp', 'accuracy_comparison.png', 'anaconda3', 'AppData', 'Application Data', 'base_case.json', 'Contacts', 'Cookies', 'DC', 'DIP_FINAL.html', 'Documents', 'Downloads', 'eclipse', 'eclipse-workspace', 'emnist-letters-mapping.txt', 'emnist-letters-train-images-idx3-ubyte.gz', 'Favorites', 'git', 'img.png', 'Links', 'Local Settings', 'Music', 'My Documents', 'NetHood', 'NTUSER.DAT', 'ntuser.dat.LOG1', 'ntuser.dat.LOG2', 'NTUSER.DAT{2ad838bb-efea-11ee-a54d-000d3a94eaa1}.TxR.0.regtrans-ms', 'NTUSER.DAT{2ad838bb-efea-11ee-a54d-000d3a94eaa1}.TxR.0.regtrans-ms.cnpf', 'NTUSER.DAT{2ad838bb-efea-11ee-a54d-000d3a94eaa1}.TxR.1.regtrans-ms', 'NTUSER.DAT{2ad838bb-efea-11ee-a54d-000d3a94eaa1}.TxR.1.regtrans-ms.cnpf', 'NTUSE

# Read the JSON file

In [3]:
import json

with open("test_case_1.json", "r") as file:
    data = json.load(file)

print(data)

{'warehouses': {'W1': [34, 29], 'W2': [95, 4], 'W3': [86, 21], 'W4': [32, 5], 'W5': [14, 12]}, 'agents': {'A1': [89, 16], 'A2': [52, 21], 'A3': [17, 17], 'A4': [99, 83]}, 'packages': [{'id': 'P1', 'warehouse': 'W5', 'destination': [12, 7]}, {'id': 'P2', 'warehouse': 'W2', 'destination': [100, 1]}, {'id': 'P3', 'warehouse': 'W5', 'destination': [24, 17]}, {'id': 'P4', 'warehouse': 'W2', 'destination': [87, 14]}, {'id': 'P5', 'warehouse': 'W5', 'destination': [6, 2]}, {'id': 'P6', 'warehouse': 'W3', 'destination': [83, 19]}, {'id': 'P7', 'warehouse': 'W5', 'destination': [10, 2]}, {'id': 'P8', 'warehouse': 'W4', 'destination': [37, 13]}, {'id': 'P9', 'warehouse': 'W1', 'destination': [44, 35]}, {'id': 'P10', 'warehouse': 'W2', 'destination': [102, 0]}, {'id': 'P11', 'warehouse': 'W5', 'destination': [7, 22]}, {'id': 'P12', 'warehouse': 'W4', 'destination': [40, 8]}]}


# Normalize warehousea

In [4]:
# NORMALIZE WAREHOUSES

# Format 1:
# [
#     {"id": "W1", "location": [0, 0]},
#     {"id": "W2", "location": [50, 75]}
# ]

# Format 2:
# {
#     "W1": [34, 29],
#     "W2": [95, 4]
# }

if isinstance(data["warehouses"], list):

    warehouse_lookup = {}

    for warehouse in data["warehouses"]:
        warehouse_lookup[warehouse["id"]] = warehouse["location"]

else:

    warehouse_lookup = data["warehouses"].copy()


# Normalize Agents

In [5]:
# NORMALIZE AGENTS
# Format 1:
# [
#     {"id": "A1", "location": [5, 5]},
#     {"id": "A2", "location": [60, 60]}
# ]

# Format 2:
# {
#     "A1": [89, 16],
#     "A2": [52, 21]
# }

if isinstance(data["agents"], list):

    agent_locations = {}

    for agent in data["agents"]:
        agent_locations[agent["id"]] = agent["location"]

else:

    agent_locations = data["agents"].copy()


# Normalize packages

In [6]:
# NORMALIZE PACKAGES
packages = []

for package in data["packages"]:

    package_id = package["id"]

    # Base case uses "warehouse_id"
    if "warehouse_id" in package:
        warehouse_id = package["warehouse_id"]

    # New JSON uses "warehouse"
    else:
        warehouse_id = package["warehouse"]

    destination = package["destination"]

    packages.append({
        "id": package_id,
        "warehouse_id": warehouse_id,
        "destination": destination
    })


# Calculate Euclidean distance

In [7]:
def calculate_distance(point1, point2):
    x1, y1 = point1
    x2, y2 = point2

    distance = math.sqrt(
        (x2 - x1) ** 2 +
        (y2 - y1) ** 2
    )

    return distance

# Example

In [8]:
print(calculate_distance([0, 0], [3, 4]))

5.0


# Find the nearest agent

In [9]:
def find_nearest_agent(
    warehouse_location,
    current_locations
):

    nearest_agent = None
    minimum_distance = float("inf")

    for agent_id, agent_location in current_locations.items():

        distance = calculate_distance(
            agent_location,
            warehouse_location
        )

        if distance < minimum_distance:

            minimum_distance = distance
            nearest_agent = agent_id

    return nearest_agent

# Create the simulation

In [10]:
def simulate_delivery():

    # Current location of each agent
    current_locations = {}

    for agent_id, location in agent_locations.items():

        current_locations[agent_id] = location.copy()


    # Report for every agent
    report = {}

    for agent_id in agent_locations:

        report[agent_id] = {
            "packages_delivered": 0,
            "total_distance": 0.0
        }


    # Process packages one by one
    for package in packages:

        package_id = package["id"]

        warehouse_id = package["warehouse_id"]

        destination = package["destination"]


        # Get warehouse location
        warehouse_location = warehouse_lookup[warehouse_id]


        # Find nearest agent from CURRENT location
        assigned_agent = find_nearest_agent(
            warehouse_location,
            current_locations
        )


        # Get current location of assigned agent
        agent_location = current_locations[assigned_agent]


        # ----------------------------------------------------
        # Agent -> Warehouse
        # ----------------------------------------------------

        distance_to_warehouse = calculate_distance(
            agent_location,
            warehouse_location
        )


        # ----------------------------------------------------
        # Warehouse -> Destination
        # ----------------------------------------------------

        distance_to_destination = calculate_distance(
            warehouse_location,
            destination
        )


        # ----------------------------------------------------
        # Total distance
        # ----------------------------------------------------

        total_package_distance = (
            distance_to_warehouse +
            distance_to_destination
        )


        # Update agent report
        report[assigned_agent]["packages_delivered"] += 1

        report[assigned_agent]["total_distance"] += (
            total_package_distance
        )


        # Agent is now at the destination
        current_locations[assigned_agent] = destination.copy()


        # Display package result
        print(
            package_id,
            "->",
            assigned_agent,
            "| Warehouse:",
            warehouse_id,
            "| Distance:",
            round(total_package_distance, 2)
        )


    return report


# Simulation Delivery

In [11]:
report = simulate_delivery()

P1 -> A3 | Warehouse: W5 | Distance: 11.22
P2 -> A1 | Warehouse: W2 | Distance: 19.25
P3 -> A3 | Warehouse: W5 | Distance: 16.57
P4 -> A1 | Warehouse: W2 | Distance: 18.64
P5 -> A3 | Warehouse: W5 | Distance: 23.99
P6 -> A1 | Warehouse: W3 | Distance: 10.68
P7 -> A3 | Warehouse: W5 | Distance: 23.58
P8 -> A3 | Warehouse: W4 | Distance: 31.64
P9 -> A3 | Warehouse: W1 | Distance: 27.94
P10 -> A1 | Warehouse: W2 | Distance: 27.27
P11 -> A3 | Warehouse: W5 | Distance: 50.01
P12 -> A2 | Warehouse: W4 | Distance: 34.16


# Calculate efficiency

In [12]:
for agent_id in report:

    packages_delivered = report[agent_id]["packages_delivered"]
    total_distance = report[agent_id]["total_distance"]

    if packages_delivered > 0:

        report[agent_id]["efficiency"] = (
            total_distance / packages_delivered
        )

    else:

        report[agent_id]["efficiency"] = 0.0

# Find the most efficient agent

In [13]:
best_agent = min(
    report,
    key=lambda agent_id: report[agent_id]["efficiency"]
)

report["best_agent"] = best_agent

# Round the values

In [14]:
for agent_id in report:

    if agent_id == "best_agent":
        continue

    report[agent_id]["total_distance"] = round(
        report[agent_id]["total_distance"],
        2
    )

    report[agent_id]["efficiency"] = round(
        report[agent_id]["efficiency"],
        2
    )

# Display the final report

In [15]:
print("\n" + "=" * 50)
print("FINAL DELIVERY REPORT")
print("=" * 50)

print(json.dumps(report, indent=4))



FINAL DELIVERY REPORT
{
    "A1": {
        "packages_delivered": 4,
        "total_distance": 75.83,
        "efficiency": 18.96
    },
    "A2": {
        "packages_delivered": 1,
        "total_distance": 34.16,
        "efficiency": 34.16
    },
    "A3": {
        "packages_delivered": 7,
        "total_distance": 184.93,
        "efficiency": 26.42
    },
    "A4": {
        "packages_delivered": 0,
        "total_distance": 0.0,
        "efficiency": 0.0
    },
    "best_agent": "A4"
}


# Save report.json

In [16]:
output_file = "report.json"

with open(output_file, "w") as file:
    json.dump(report, file, indent=4)

print("\nReport saved successfully as:", output_file)
print(output_file)


Report saved successfully as: report.json
report.json


# Verify the saved file

In [17]:
import os

print(os.path.exists("report.json"))

True


In [18]:
with open("report.json", "r") as file:
    saved_report = json.load(file)

print(json.dumps(saved_report, indent=4))

{
    "A1": {
        "packages_delivered": 4,
        "total_distance": 75.83,
        "efficiency": 18.96
    },
    "A2": {
        "packages_delivered": 1,
        "total_distance": 34.16,
        "efficiency": 34.16
    },
    "A3": {
        "packages_delivered": 7,
        "total_distance": 184.93,
        "efficiency": 26.42
    },
    "A4": {
        "packages_delivered": 0,
        "total_distance": 0.0,
        "efficiency": 0.0
    },
    "best_agent": "A4"
}
